# Housing prices per m² and 2026 municipal election results

**Second lens on the data:** does the median property price per m² in a commune correlate with which political bloc won there?

**Source:** DVF (Demandes de Valeurs Foncières) 2024 — Ministère de l'Économie  
**Election data:** same 937-commune dataset from notebook 01

**Pipeline:** DVF transactions → filter residential sales → compute prix/m² → aggregate to commune level → join election results → visualise by bloc

## 1. Load DVF raw data

Raw transaction file: ~3.5M rows, pipe-separated (`|`). We load only the 6 columns we need — the file is 460 MB so columnar loading keeps memory manageable. We'll filter and clean in the next cell.

In [ ]:
import pandas as pd

# Only load the columns we need — avoids pulling 460 MB fully into memory
cols = ['Nature mutation', 'Type local', 'Code departement', 'Code commune',
        'Valeur fonciere', 'Surface reelle bati']

dvf = pd.read_csv('../data/raw/ValeursFoncieres-2025.txt', sep='|',
                  usecols=cols, low_memory=False)

print(f"Loaded: {len(dvf):,} rows")
print(dvf.dtypes)

## 2. Filter to residential sales and compute prix/m²

We keep only standard sales (`Vente`) of houses and apartments. This excludes:
- Dependencies (caves, garages) — no meaningful price/m²
- Commercial premises — different market
- Off-plan sales (VEFA) and exchanges — price not directly comparable

DVF stores monetary values with a French comma decimal (`"346,50"`), so we convert before dividing. We cap at €20,000/m² — anything above this is almost certainly a data entry error (e.g. a surface recorded as 1 m² instead of 100 m²).

In [ ]:
# Keep standard residential sales only — houses and apartments
dvf = dvf[
    (dvf['Nature mutation'] == 'Vente') &
    (dvf['Type local'].isin(['Appartement', 'Maison']))
].copy()

print(f"After filter: {len(dvf):,} rows")

# DVF uses French decimal format: "346,50" → 346.50
dvf['valeur'] = pd.to_numeric(
    dvf['Valeur fonciere'].str.replace(',', '.', regex=False),
    errors='coerce'
)
dvf['surface'] = pd.to_numeric(dvf['Surface reelle bati'], errors='coerce')

# Drop rows where surface or price is missing or zero — can't compute prix/m²
dvf = dvf[(dvf['surface'] > 0) & (dvf['valeur'] > 0)].copy()

dvf['prix_m2'] = dvf['valeur'] / dvf['surface']

# Hard cap at €20,000/m² — above this indicates a recording error, not a real price
dvf = dvf[dvf['prix_m2'] < 20_000].copy()
print(f"After prix_m2 cap: {len(dvf):,} rows")

print(dvf['prix_m2'].describe())

## 3. Reconstruct the 5-character INSEE commune code

DVF stores département and commune as separate fields. We need to join on the standard 5-character INSEE code used in our election data (e.g. `01160`, `75056`).

- Département: 1 or 2 digits in DVF → zero-pad to 2 characters
- Commune: 1–3 digits → zero-pad to 3 characters
- Exception: Corse uses `2A`/`2B` as département codes — these pass through correctly since we're working with strings

In [ ]:
# Cast to string and zero-pad to produce the 5-char INSEE code
dvf['dept_str'] = dvf['Code departement'].astype(str).str.strip().str.zfill(2)
dvf['comm_str'] = dvf['Code commune'].astype(str).str.strip().str.zfill(3)
dvf['code_commune'] = dvf['dept_str'] + dvf['comm_str']

print("Sample code_commune values:")
print(dvf['code_commune'].head(10).tolist())
print(f"Unique communes in DVF: {dvf['code_commune'].nunique():,}")

## 4. Aggregate to commune level: median prix/m²

One row per commune. We use **median** rather than mean — property prices are right-skewed, and median is less sensitive to the occasional luxury transaction that can inflate a commune's average.

We also keep `n_transactions` as a quality signal: communes with very few transactions (n < 5) have less reliable medians.

In [ ]:
# One row per commune: median price and transaction count
dvf_commune = dvf.groupby('code_commune').agg(
    median_prix_m2=('prix_m2', 'median'),
    n_transactions=('prix_m2', 'count')
).reset_index()

print(f"Communes with prix data: {len(dvf_commune):,}")
print(dvf_commune.describe())

## 5. Rebuild the election dataset and join

We reconstruct `df_clean` using the same logic as notebook 01: load the commune-level parquet, derive `abstention_rate`, find the winning list by vote share, then join to `nuances.csv` to get the `winning_bloc` column.

This keeps the notebook self-contained — no dependency on notebook 01's kernel state.

In [ ]:
# Load commune-level election parquet
df = pd.read_parquet('../data/raw/commune.parquet')

# Derive abstention rate from raw percentage string (e.g. "56,35%")
df['abstention_rate'] = df['% Abstentions'].str.replace('%', '').str.replace(',', '.').astype(float)

# Convert vote share columns for lists 1–5 to float
voix_cols = [f'% Voix/exprimés {i}' for i in range(1, 6)]
for col in voix_cols:
    df[col] = pd.to_numeric(
        df[col].str.replace('%', '').str.replace(',', '.'),
        errors='coerce'
    )

# Identify which list got the highest vote share in each commune
winning_col = df[voix_cols].idxmax(axis=1)
winning_num = winning_col.str.split(' ').str[-1]  # extract list number from column name

# Look up the nuance code for the winning list in each row
df['winning_nuance'] = [
    row[f'Nuance liste {num}']
    for (_, row), num in zip(df.iterrows(), winning_num)
]

# Join nuances.csv to get bloc grouping (EXG, GAU, CENT, DIV, DTE, EXD)
nuances = pd.read_csv('../data/raw/nuances.csv', sep=';')
nuances_liste = nuances[nuances['type_nuance'] == 'liste']
df = df.merge(
    nuances_liste[['libelle_nuance', 'bloc']].rename(columns={'bloc': 'winning_bloc'}),
    left_on='winning_nuance', right_on='libelle_nuance', how='left'
)

# Drop communes where we couldn't assign a bloc (unmatched nuances)
df_clean = df[df['winning_bloc'].notna()].copy()
print(f"df_clean: {df_clean.shape}")

## 6. Join election results with DVF housing prices

Left join on `Code commune` — we keep all 937 election communes. Communes with no DVF transactions in 2024 will have `NaN` for price (102 communes, mostly very small rural ones with no recorded sales).

**Note on Paris, Lyon, Marseille:** DVF records transactions at arrondissement level (`75101`–`75120`, `69381`–`69389`, `13201`–`13216`), not at the main commune code (`75056`, `69123`, `13055`) used in the election data. These three cities have no DVF match and are excluded from the export — 835 communes total.

In [ ]:
# Left join — keeps all 937 election communes, NaN where no DVF data exists
df_prix = df_clean.merge(
    dvf_commune,
    left_on='Code commune',
    right_on='code_commune',
    how='left'
)

matched = df_prix['median_prix_m2'].notna().sum()
print(f"Communes with prix data: {matched} / {len(df_prix)}")
print(f"Missing: {df_prix['median_prix_m2'].isna().sum()}")

df_prix[['Code commune', 'Libellé commune', 'abstention_rate',
         'winning_bloc', 'median_prix_m2', 'n_transactions']].head(10)

## 7. Sanity check: median prix/m² by winning bloc

A quick aggregation before visualising. We expect DTE and CENT blocs to skew higher (suburban/peri-urban homeowner communes), and EXD potentially lower (smaller towns, post-industrial areas). EXG has very few communes (n=6) so interpret with caution.

In [ ]:
# Median and mean prix/m² by winning bloc — first look at whether a signal exists
bloc_prix = df_prix.groupby('winning_bloc').agg(
    communes=('Code commune', 'count'),
    median_prix=('median_prix_m2', 'median'),
    mean_prix=('median_prix_m2', 'mean')
).sort_values('median_prix', ascending=False)

print(bloc_prix.to_string())

The summary table confirms a signal exists. We export the data first (section 8), then plot the full distribution as a box plot (section 9) to see spread, not just central tendency.

## 7b. Paris, Lyon, Marseille — aggregate from arrondissement data

These three cities use arrondissement-level INSEE codes in DVF (`75101`–`75120`, `69381`–`69389`, `13201`–`13216`) but a single commune code in the election data (`75056`, `69123`, `13055`). We pool all arrondissement transactions, take the median, then patch `df_prix` directly before export.

In [ ]:
plm_ranges = {
    '75056': {'nom': 'Paris',     'codes': [f'75{str(i).zfill(3)}' for i in range(101, 121)]},
    '69123': {'nom': 'Lyon',      'codes': [f'69{str(i).zfill(3)}' for i in range(381, 390)]},
    '13055': {'nom': 'Marseille', 'codes': [f'13{str(i).zfill(3)}' for i in range(201, 217)]},
}

# Pool all arrondissement transactions for each city and take the median
for code_commune, meta in plm_ranges.items():
    transactions = dvf[dvf['code_commune'].isin(meta['codes'])]
    if len(transactions) == 0:
        print(f"WARNING: no DVF transactions found for {meta['nom']}")
        continue

    median_prix = transactions['prix_m2'].median()
    n = len(transactions)

    # Patch the row in df_prix (matched by main commune code, not arrondissement)
    mask = df_prix['Code commune'] == code_commune
    df_prix.loc[mask, 'median_prix_m2'] = median_prix
    df_prix.loc[mask, 'n_transactions'] = n
    print(f"{meta['nom']}: {n:,} transactions -> median {median_prix:,.0f} euros/m2")

print(f"Total communes with prix data after PLM patch: {df_prix['median_prix_m2'].notna().sum()}")

## 8. JSON export for web visualisation

Export the 838 communes with price data (835 matched directly + Paris, Lyon, Marseille via arrondissement aggregation) to a clean JSON file. This is what the Plotly.js web viz will consume — lives at `data/processed/` and committed to the repo (raw CSVs are gitignored).

In [ ]:
import json, os

os.makedirs('../data/processed', exist_ok=True)

# Include PLM — df_prix now has median_prix_m2 patched for Paris, Lyon, Marseille
export_cols = ['Code commune', 'Libellé commune', 'abstention_rate',
               'winning_bloc', 'winning_nuance', 'median_prix_m2', 'n_transactions']
df_viz = df_prix[df_prix['median_prix_m2'].notna()].copy()

df_export = df_viz[export_cols].copy()

# Rename to snake_case for clean JSON keys
df_export.columns = ['code_commune', 'nom_commune', 'abstention_rate',
                     'winning_bloc', 'winning_nuance', 'median_prix_m2', 'n_transactions']

output_path = '../data/processed/prix-logement-elections-2025.json'
df_export.to_json(output_path, orient='records', force_ascii=False, indent=2)

print(f"Exported {len(df_export)} communes to {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

## 9. Visualisation: prix/m² distribution by winning bloc

Box plot with a log y-axis — price distributions are right-skewed across all blocs, so log scale makes the spread readable without high-price outliers dominating. Triangles mark the highest- and lowest-priced commune per bloc, labelled by name.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Canonical bloc colours (Le Monde conventions)
bloc_colors = {
    'EXG': '#cc0000',
    'GAU': '#e75480',
    'CENT': '#ffcc00',
    'DTE': '#0055a4',
    'EXD': '#1a1a2e',
    'DIV': '#aaaaaa',
}

# Full French labels for the legend
bloc_labels = {
    'EXG': 'Extrême gauche',
    'GAU': 'Gauche',
    'CENT': 'Centre',
    'DIV': 'Divers',
    'DTE': 'Droite',
    'EXD': 'Extrême droite',
}

# Left-to-right political order on the x-axis
bloc_order = ['EXG', 'GAU', 'CENT', 'DIV', 'DTE', 'EXD']
nom_col = 'Libellé commune'

# Box plot — log_y because price distributions are right-skewed
fig2 = px.box(
    df_viz,
    x='winning_bloc',
    y='median_prix_m2',
    color='winning_bloc',
    color_discrete_map=bloc_colors,
    log_y=True,
    category_orders={'winning_bloc': bloc_order},
    labels={
        'winning_bloc': 'Bloc',
        'median_prix_m2': 'Prix médian au m² (€, échelle log)',
    },
    title='Distribution des prix au m² par bloc politique — Municipales 2026',
)

# Replace short bloc codes in the legend with full French names
for trace in fig2.data:
    bloc = trace.name
    trace.name = bloc_labels.get(bloc, bloc)

fig2.update_layout(
    showlegend=True,
    legend_title_text='Bloc politique',
    plot_bgcolor='white',
    paper_bgcolor='white',
    yaxis=dict(showgrid=True, gridcolor='#eeeeee'),
    margin=dict(r=160),  # extra right margin to avoid clipping commune name labels
)

# Add named min/max commune markers as scatter traces on top of each box
for bloc in bloc_order:
    subset = df_viz[df_viz['winning_bloc'] == bloc].dropna(subset=['median_prix_m2'])
    if len(subset) == 0:
        continue

    max_row = subset.loc[subset['median_prix_m2'].idxmax()]
    min_row = subset.loc[subset['median_prix_m2'].idxmin()]
    color = bloc_colors[bloc]

    # Highest-priced commune — triangle-up above the box
    fig2.add_trace(go.Scatter(
        x=[bloc],
        y=[max_row['median_prix_m2']],
        mode='markers+text',
        marker=dict(symbol='triangle-up', size=8, color=color),
        text=[max_row[nom_col]],
        textposition='top center',
        textfont=dict(size=8, color='#333333'),
        showlegend=False,
        hoverinfo='skip',
    ))

    # Lowest-priced commune — triangle-down below the box
    fig2.add_trace(go.Scatter(
        x=[bloc],
        y=[min_row['median_prix_m2']],
        mode='markers+text',
        marker=dict(symbol='triangle-down', size=8, color=color),
        text=[min_row[nom_col]],
        textposition='bottom center',
        textfont=dict(size=8, color='#333333'),
        showlegend=False,
        hoverinfo='skip',
    ))

fig2.show()